# AgentDefinitions in Claude Agent SDK

This notebook demonstrates how to use `AgentDefinition` from the **Claude Agent SDK** (`claude-agent-sdk`) to build multi-agent workflows with Claude.

`AgentDefinition` lets you register named **subagents** — each with its own system prompt, tool access, and description — that Claude's orchestrator can spawn on demand.

---

### Prerequisites
- Python 3.10+
- Node.js ≥ 18 (Claude Code CLI dependency)
- An Anthropic API key set as `ANTHROPIC_API_KEY`

## 1. Installation

In [ ]:
# Install the Claude Agent SDK
!pip install claude-agent-sdk python-dotenv

## 2. Environment Setup

In [ ]:
import os
import asyncio

# Option A: Load from a .env file (recommended)
from dotenv import load_dotenv
load_dotenv(".env")

# Option B: Set directly in-notebook (not recommended for shared notebooks)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # Replace with your key

print("API key loaded:", "ANTHROPIC_API_KEY" in os.environ)

## 3. Imports

In [ ]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    AssistantMessage,
    TextBlock,
    ResultMessage,
    ToolUseBlock,
)

print("Imports successful ✅")

## 4. What Is `AgentDefinition`?

`AgentDefinition` is a **dataclass** that describes a subagent Claude can spawn. It contains:

| Field | Type | Description |
|---|---|---|
| `description` | `str` | Brief description of what this subagent does (used by the orchestrator to decide when to call it) |
| `prompt` | `str` | System prompt that governs the subagent's behavior |
| `tools` | `list[str]` | Tools the subagent is allowed to use (e.g. `"Read"`, `"Glob"`, `"Bash"`) |

> **Note:** `AgentDefinition` uses **camelCase field names** internally (unlike `ClaudeAgentOptions` which uses snake_case). Always pass keyword arguments by name.

In [ ]:
# Inspect AgentDefinition
import dataclasses

for field in dataclasses.fields(AgentDefinition):
    print(f"  {field.name}: {field.type}")

## 5. Helper: Message Parser

A reusable function to extract and print responses from the async iterator returned by `query()`.

In [ ]:
async def stream_response(prompt: str, options: ClaudeAgentOptions, verbose: bool = False):
    """Run a query and return the final result text."""
    result_text = None
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock) and verbose:
                    print(f"  🔧 Tool call: {block.name} | input: {block.input}")
                elif isinstance(block, TextBlock) and verbose:
                    print(f"  💬 Assistant: {block.text[:200]}")
        elif isinstance(message, ResultMessage):
            result_text = message.result
    return result_text

print("Helper defined ✅")

## 6. Basic Example: Single Subagent

Here we define one subagent — a **code reviewer** — and ask the orchestrator to use it.

In [ ]:
async def demo_single_subagent():
    # Define a code-reviewer subagent
    code_reviewer = AgentDefinition(
        description="Expert code reviewer. Reviews Python code for quality, bugs, and best practices.",
        prompt=(
            "You are a senior Python engineer and code reviewer. "
            "When given code, analyze it for correctness, style (PEP 8), "
            "potential bugs, and improvements. Be concise and actionable."
        ),
        tools=["Read", "Glob", "Grep"],
    )

    options = ClaudeAgentOptions(
        # Enable the Agent tool so the orchestrator can spawn subagents
        allowed_tools=["Agent"],
        agents={
            "code-reviewer": code_reviewer,
        },
        max_turns=5,
        model="claude-sonnet-4-5",
    )

    sample_code = '''
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total = total + n
    return total / len(numbers)
    '''

    prompt = (
        f"Use the code-reviewer agent to review this Python function and suggest improvements:\n"
        f"```python{sample_code}```"
    )

    print("Running code reviewer subagent...\n")
    result = await stream_response(prompt, options, verbose=True)
    print("\n--- FINAL RESULT ---")
    print(result)

asyncio.run(demo_single_subagent())

## 7. Multiple Subagents: Orchestrator Pattern

Register multiple specialised subagents. The orchestrator decides which ones to call based on their `description`.

In [ ]:
async def demo_multiple_subagents():
    agents = {
        # --- Subagent 1: Security Auditor ---
        "security-auditor": AgentDefinition(
            description="Scans code for security vulnerabilities such as SQL injection, XSS, and insecure dependencies.",
            prompt=(
                "You are a cybersecurity expert specializing in application security. "
                "Identify security vulnerabilities in the provided code. "
                "Reference OWASP Top 10 where applicable and suggest secure alternatives."
            ),
            tools=["Read", "Grep"],
        ),

        # --- Subagent 2: Performance Analyst ---
        "performance-analyst": AgentDefinition(
            description="Analyzes code for performance bottlenecks, inefficient algorithms, and memory issues.",
            prompt=(
                "You are a performance engineering expert. "
                "Profile the provided code for time/space complexity issues, "
                "inefficient loops, excessive memory usage, and optimization opportunities. "
                "Provide Big-O analysis where relevant."
            ),
            tools=["Read"],
        ),

        # --- Subagent 3: Documentation Writer ---
        "doc-writer": AgentDefinition(
            description="Writes clear docstrings and inline documentation for Python code.",
            prompt=(
                "You are a technical writer specializing in Python documentation. "
                "Write Google-style docstrings for all functions/classes provided. "
                "Include Args, Returns, Raises, and a usage Example in each docstring."
            ),
            tools=[],
        ),
    }

    options = ClaudeAgentOptions(
        allowed_tools=["Agent"],
        agents=agents,
        max_turns=10,
        model="claude-sonnet-4-5",
        system_prompt=(
            "You are an expert orchestrator. For each user request, "
            "identify which specialist agents should be invoked and synthesize their results into a unified report."
        ),
    )

    code_snippet = '''
import sqlite3

def get_user(username):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    # WARNING: Direct string interpolation — SQL injection risk!
    query = f"SELECT * FROM users WHERE username = \'{username}\'"
    cursor.execute(query)
    results = []
    row = cursor.fetchone()
    while row is not None:
        results.append(row)
        row = cursor.fetchone()
    return results
    '''

    prompt = (
        "Run a full analysis of this code using the security-auditor, "
        "performance-analyst, and doc-writer agents. Then produce a consolidated report.\n"
        f"```python{code_snippet}```"
    )

    print("Running multi-agent analysis pipeline...\n")
    result = await stream_response(prompt, options, verbose=True)
    print("\n========== CONSOLIDATED REPORT ==========")
    print(result)

asyncio.run(demo_multiple_subagents())

## 8. Subagent With File Tools

Subagents can be given real filesystem tools like `Read`, `Glob`, `Grep`, `Write`, and `Bash`.
Here, a **research agent** scans a local project directory.

In [ ]:
import tempfile
import os

async def demo_file_aware_subagent():
    # Create a temporary project directory with sample files
    with tempfile.TemporaryDirectory() as tmpdir:
        # Write sample Python files
        for fname, content in [
            ("main.py",   "# Entry point\nfrom utils import greet\nprint(greet('World'))\n"),
            ("utils.py",  "def greet(name):\n    return f'Hello, {name}!'\n"),
            ("config.py", "DEBUG = True\nSECRET_KEY = 'hardcoded_secret_123'\n"),  # intentional bad practice
        ]:
            with open(os.path.join(tmpdir, fname), "w") as f:
                f.write(content)

        agents = {
            "file-scanner": AgentDefinition(
                description="Scans Python source files in the working directory for hardcoded secrets and debug flags.",
                prompt=(
                    "You are a security scanner. Use Grep and Read tools to scan Python files "
                    "for hardcoded secrets, DEBUG=True, passwords, or API keys. "
                    "Report each finding with the filename and line content."
                ),
                tools=["Read", "Glob", "Grep"],
            )
        }

        options = ClaudeAgentOptions(
            allowed_tools=["Agent"],
            agents=agents,
            max_turns=8,
            cwd=tmpdir,  # Point Claude to the temp project directory
            permission_mode="acceptEdits",
        )

        result = await stream_response(
            "Use the file-scanner agent to scan all Python files in the current directory for security issues.",
            options,
            verbose=True,
        )
        print("\n--- SCAN RESULT ---")
        print(result)

asyncio.run(demo_file_aware_subagent())

## 9. Tracking Subagent Messages via `parent_tool_use_id`

Messages produced *inside* a subagent's context carry a `parent_tool_use_id` field that links them back to the originating `Agent` tool call. This enables fine-grained tracing.

In [ ]:
async def demo_trace_subagent_messages():
    agents = {
        "summarizer": AgentDefinition(
            description="Summarizes any text provided into three bullet points.",
            prompt="You are a concise summarizer. Always respond with exactly 3 bullet points.",
            tools=[],
        )
    }

    options = ClaudeAgentOptions(
        allowed_tools=["Agent"],
        agents=agents,
        max_turns=5,
    )

    text = (
        "Machine learning is a subset of artificial intelligence that enables systems to learn "
        "and improve from experience without being explicitly programmed. It focuses on developing "
        "computer programs that can access data and use it to learn for themselves."
    )

    print("Streaming all messages with trace info:\n")
    async for message in query(
        prompt=f"Use the summarizer agent to summarize this: {text}",
        options=options,
    ):
        msg_type = type(message).__name__
        parent_id = getattr(message, "parent_tool_use_id", None)
        origin = f" [subagent of: {parent_id}]" if parent_id else " [orchestrator]"
        print(f"  📨 {msg_type}{origin}")

        if isinstance(message, ResultMessage):
            print(f"\n--- FINAL RESULT ---\n{message.result}")

asyncio.run(demo_trace_subagent_messages())

## 10. Sequential Subagent Pipeline

Chain subagents where the output of one feeds the next. Here: **Researcher → Writer → Editor**.

In [ ]:
async def run_agent_step(prompt: str, agents: dict, model: str = "claude-haiku-4-5") -> str:
    """Run a single step of the pipeline."""
    options = ClaudeAgentOptions(
        allowed_tools=["Agent"],
        agents=agents,
        max_turns=5,
        model=model,
    )
    return await stream_response(prompt, options)


async def demo_sequential_pipeline():
    topic = "the benefits of test-driven development"

    # Step 1: Researcher gathers key points
    researcher_agents = {
        "researcher": AgentDefinition(
            description="Gathers and organises key facts about a given topic.",
            prompt="You are a research analyst. Produce 5 concise, factual bullet points about the given topic.",
            tools=[],
        )
    }
    print("Step 1: Researching...")
    research_output = await run_agent_step(
        f"Use the researcher agent to gather key points about: {topic}",
        researcher_agents,
    )
    print(f"Research output:\n{research_output}\n")

    # Step 2: Writer drafts a paragraph from the research
    writer_agents = {
        "writer": AgentDefinition(
            description="Drafts a clear, engaging paragraph from bullet-point research notes.",
            prompt="You are a technical writer. Turn the provided bullet points into a single coherent paragraph.",
            tools=[],
        )
    }
    print("Step 2: Writing...")
    draft_output = await run_agent_step(
        f"Use the writer agent to draft a paragraph from these notes:\n{research_output}",
        writer_agents,
    )
    print(f"Draft:\n{draft_output}\n")

    # Step 3: Editor polishes the draft
    editor_agents = {
        "editor": AgentDefinition(
            description="Proofreads and improves grammar, clarity, and conciseness of a text draft.",
            prompt="You are a professional editor. Polish the draft for grammar, clarity, and flow. Return only the final edited version.",
            tools=[],
        )
    }
    print("Step 3: Editing...")
    final_output = await run_agent_step(
        f"Use the editor agent to polish this draft:\n{draft_output}",
        editor_agents,
    )
    print(f"\n========== FINAL POLISHED OUTPUT ==========\n{final_output}")

asyncio.run(demo_sequential_pipeline())

## 11. Key Concepts Summary

| Concept | Detail |
|---|---|
| `AgentDefinition` | Dataclass describing a subagent (description, prompt, tools) |
| `agents` param | Dict in `ClaudeAgentOptions` mapping names → `AgentDefinition` instances |
| `"Agent"` tool | Must be in `allowed_tools` for the orchestrator to invoke subagents |
| `parent_tool_use_id` | Links subagent messages back to their parent tool call for tracing |
| `cwd` | Set the working directory for file-system-aware agents |
| `permission_mode` | `"acceptEdits"` auto-approves writes; default requires confirmation |
| Orchestrator model | Controlled by `model` in `ClaudeAgentOptions`; subagents inherit unless overridden |

### Architecture Overview
```
User Prompt
    │
    ▼
┌─────────────────────────────────┐
│  Orchestrator (Claude)          │   ← ClaudeAgentOptions.system_prompt
│  allowed_tools: ["Agent", ...]  │
└────────────┬────────────────────┘
             │  calls Agent tool
     ┌───────┴────────┐
     ▼                ▼
┌──────────┐   ┌──────────────┐
│ Subagent │   │  Subagent    │   ← Each has its own AgentDefinition
│ "writer" │   │ "reviewer"   │       (prompt, tools, description)
└──────────┘   └──────────────┘
```

## 12. Further Resources

- 📖 [Claude Agent SDK Overview](https://code.claude.com/docs/en/agent-sdk/overview)
- 📖 [Python SDK API Reference](https://code.claude.com/docs/en/agent-sdk/python)
- 📖 [Quickstart Guide](https://platform.claude.com/docs/en/agent-sdk/quickstart)
- 💻 [GitHub: anthropics/claude-agent-sdk-python](https://github.com/anthropics/claude-agent-sdk-python)